In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from zipfile import ZipFile

zip_path = "/content/drive/MyDrive/output.zip"
extract_path = "/content"

with ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Zip açıldı ✅")

Zip açıldı ✅


In [6]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# PATH
DATA_DIR = "/content/output"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15

# MODEL PATH
BEST_MODEL_PATH = "/content/drive/MyDrive/best_model.h5"
FINAL_MODEL_PATH = "/content/drive/MyDrive/final_model.h5"

# SINIFLAR
classes = sorted([
    d for d in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, d))
])

num_classes = len(classes)

print("Sınıf sayısı:", num_classes)
print(classes)
print("GPU:", tf.config.list_physical_devices("GPU"))

# DATA AUGMENTATION
train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    rotation_range=25,
    zoom_range=0.25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input
)

# GENERATOR
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    classes=classes,
    class_mode="categorical",
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    classes=classes,
    class_mode="categorical",
    shuffle=False
)

# ÖNCEKİ MODELİ YÜKLE
model = keras.models.load_model(BEST_MODEL_PATH)

# COMPILE
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# CALLBACKS
checkpoint = keras.callbacks.ModelCheckpoint(
    BEST_MODEL_PATH,
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

# TRAIN - 8. EPOCH'TAN DEVAM
history = model.fit(
    train_generator,
    validation_data=val_generator,
    initial_epoch=7,
    epochs=EPOCHS,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

# SAVE FINAL
model.save(FINAL_MODEL_PATH)

print("Best model:", BEST_MODEL_PATH)
print("Final model:", FINAL_MODEL_PATH)

Sınıf sayısı: 35
['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'No_leaf', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy']
GPU: [PhysicalDevice(na

Epoch 8/15
  52/1770 ━━━━━━━━━━━━━━━━━━━━ 13:12 461ms/step - accuracy: 0.9025 - loss: 0.3383

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


1770/1770 ━━━━━━━━━━━━━━━━━━━━ 0s 495ms/step - accuracy: 0.9108 - loss: 0.2746
Epoch 8: val_loss improved from None to 0.14040, saving model to /content/drive/MyDrive/best_model.h5



Epoch 8: finished saving model to /content/drive/MyDrive/best_model.h5
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 943s 526ms/step - accuracy: 0.9111 - loss: 0.2723 - val_accuracy: 0.9560 - val_loss: 0.1404 - learning_rate: 5.0000e-04
Epoch 9/15
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 0s 471ms/step - accuracy: 0.9147 - loss: 0.2608
Epoch 9: val_loss did not improve from 0.14040
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 872s 493ms/step - accuracy: 0.9134 - loss: 0.2625 - val_accuracy: 0.9532 - val_loss: 0.1444 - learning_rate: 5.0000e-04
Epoch 10/15
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 0s 470ms/step - accuracy: 0.9139 - loss: 0.2662
Epoch 10: val_loss improved from 0.14040 to 0.13092, saving model to /content/drive/MyDrive/best_model.h5



Epoch 10: finished saving model to /content/drive/MyDrive/best_model.h5
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 872s 493ms/step - accuracy: 0.9133 - loss: 0.2671 - val_accuracy: 0.9597 - val_loss: 0.1309 - learning_rate: 5.0000e-04
Epoch 11/15
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 0s 467ms/step - accuracy: 0.9182 - loss: 0.2529
Epoch 11: val_loss did not improve from 0.13092
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 866s 489ms/step - accuracy: 0.9166 - loss: 0.2559 - val_accuracy: 0.9555 - val_loss: 0.1377 - learning_rate: 5.0000e-04
Epoch 12/15
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 0s 479ms/step - accuracy: 0.9187 - loss: 0.2447
Epoch 12: val_loss did not improve from 0.13092

Epoch 12: ReduceLROnPlateau reducing learning rate to 0.0001500000071246177.
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 887s 501ms/step - accuracy: 0.9162 - loss: 0.2544 - val_accuracy: 0.9605 - val_loss: 0.1330 - learning_rate: 5.0000e-04
Epoch 13/15
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 0s 471ms/step - accuracy: 0.9197 - loss: 0.2427
Epoch 13: val_loss improved


Epoch 13: finished saving model to /content/drive/MyDrive/best_model.h5
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 878s 496ms/step - accuracy: 0.9231 - loss: 0.2338 - val_accuracy: 0.9631 - val_loss: 0.1196 - learning_rate: 1.5000e-04
Epoch 14/15
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 0s 481ms/step - accuracy: 0.9267 - loss: 0.2207
Epoch 14: val_loss improved from 0.11958 to 0.11439, saving model to /content/drive/MyDrive/best_model.h5



Epoch 14: finished saving model to /content/drive/MyDrive/best_model.h5
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 891s 503ms/step - accuracy: 0.9258 - loss: 0.2230 - val_accuracy: 0.9633 - val_loss: 0.1144 - learning_rate: 1.5000e-04
Epoch 15/15
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 0s 479ms/step - accuracy: 0.9285 - loss: 0.2188
Epoch 15: val_loss improved from 0.11439 to 0.10938, saving model to /content/drive/MyDrive/best_model.h5



Epoch 15: finished saving model to /content/drive/MyDrive/best_model.h5
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 888s 502ms/step - accuracy: 0.9277 - loss: 0.2203 - val_accuracy: 0.9648 - val_loss: 0.1094 - learning_rate: 1.5000e-04
Restoring model weights from the end of the best epoch: 15.


Best model: /content/drive/MyDrive/best_model.h5
Final model: /content/drive/MyDrive/final_model.h5


In [7]:
# BEST MODEL YÜKLE
model = keras.models.load_model("/content/drive/MyDrive/best_model.h5")

# MobileNetV2 base model'i bul
base_model = model.layers[0]

# Son katmanları eğitime aç
base_model.trainable = True

# İlk katmanları dondur, son 30 katmanı aç
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Düşük learning rate ile compile et
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Fine-tuning
history_fine = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

# Kaydet
model.save("/content/drive/MyDrive/fine_tuned_model.h5")

Epoch 1/5
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 0s 488ms/step - accuracy: 0.8593 - loss: 0.4746
Epoch 1: val_loss did not improve from 0.10938
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 931s 515ms/step - accuracy: 0.8805 - loss: 0.3931 - val_accuracy: 0.9565 - val_loss: 0.1417 - learning_rate: 1.0000e-05
Epoch 2/5
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 0s 484ms/step - accuracy: 0.9151 - loss: 0.2784
Epoch 2: val_loss did not improve from 0.10938

Epoch 2: ReduceLROnPlateau reducing learning rate to 2.9999999242136253e-06.
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 895s 506ms/step - accuracy: 0.9185 - loss: 0.2643 - val_accuracy: 0.9628 - val_loss: 0.1237 - learning_rate: 1.0000e-05
Epoch 3/5
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 0s 480ms/step - accuracy: 0.9238 - loss: 0.2320
Epoch 3: val_loss did not improve from 0.10938
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 888s 502ms/step - accuracy: 0.9258 - loss: 0.2269 - val_accuracy: 0.9660 - val_loss: 0.1099 - learning_rate: 3.0000e-06
Epoch 4/5
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 0s 482ms/step - accu


Epoch 4: finished saving model to /content/drive/MyDrive/best_model.h5
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 894s 505ms/step - accuracy: 0.9321 - loss: 0.2112 - val_accuracy: 0.9674 - val_loss: 0.1059 - learning_rate: 3.0000e-06
Epoch 5/5
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 0s 486ms/step - accuracy: 0.9353 - loss: 0.2020
Epoch 5: val_loss improved from 0.10592 to 0.10291, saving model to /content/drive/MyDrive/best_model.h5



Epoch 5: finished saving model to /content/drive/MyDrive/best_model.h5
1770/1770 ━━━━━━━━━━━━━━━━━━━━ 901s 509ms/step - accuracy: 0.9342 - loss: 0.2034 - val_accuracy: 0.9681 - val_loss: 0.1029 - learning_rate: 3.0000e-06
Restoring model weights from the end of the best epoch: 5.


In [8]:
converter = tf.lite.TFLiteConverter.from_keras_model(
    keras.models.load_model("/content/drive/MyDrive/best_model.h5")
)

tflite_model = converter.convert()

with open("/content/drive/MyDrive/model.tflite", "wb") as f:
    f.write(tflite_model)

Saved artifact at '/tmp/tmpnxdwiksx'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 35), dtype=tf.float32, name=None)
Captures:
  133898939013008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133898255869008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133898255867280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133898939013968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133898939013392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133898255859984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133898255868240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133898255867856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133898255869776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133898255867664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133898255869

In [10]:
classes = sorted(os.listdir(TRAIN_DIR))